# LeWorldModel — Training on Colab T4

Everything runs **locally on the Colab instance** (no Drive mount).

**Pipeline :**
1. Check GPU
2. Clone repo
3. Generate simple pendulum dataset
4. Train LeWorldModel (JEPA + SIGReg)
5. Plot loss curves
6. Visualise latent space (PCA)
7. Download best checkpoint

**Architecture :** JEPA pur — encodeur online + target encoder EMA, prédiction dans l'espace latent,  
zéro supervision pixel. Le MLP de transition force ω (vitesse angulaire) dans z.

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. GPU check

In [ ]:
import subprocess, torch

print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
print(f'PyTorch {torch.__version__}  |  CUDA {torch.version.cuda}  |  device: {torch.cuda.get_device_name(0)}')
assert torch.cuda.is_available(), 'No GPU found — switch to T4 runtime first.'

## 2. Clone repo

In [ ]:
import os

REPO_URL = 'https://github.com/JulesV19/double-pendule-wolrdmodel.git'
REPO_DIR = '/content/WorldModel'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

## 3. Install dependencies

In [ ]:
!pip install -q Pillow matplotlib numpy
print('Dependencies ready.')

## 4. Generate dataset

Generates **2 000 trajectories × 500 frames** (64×64 px) of a simple pendulum.  
Each trajectory has random initial angle **and** random initial velocity.  
~5-10 min on Colab CPU.

During training, a random **window of 100 frames** is sampled from each 500-frame
trajectory at every epoch — giving 5× more temporal diversity than a fixed dataset.

In [ ]:
from pathlib import Path

DATASET_DIR = 'dataset/pendulum'
N_TRAJ      = 2000
N_FRAMES    = 500
IMG_SIZE    = 64

existing = len(list(Path(DATASET_DIR).glob('traj_*.npz'))) if Path(DATASET_DIR).exists() else 0

if existing >= N_TRAJ:
    print(f'Dataset already present ({existing} trajectories) — skipping generation.')
else:
    print(f'Generating {N_TRAJ} trajectories × {N_FRAMES} frames…')
    from generate_dataset import generate_dataset
    generate_dataset(
        n_trajectories=N_TRAJ,
        n_frames=N_FRAMES,
        img_size=IMG_SIZE,
        dt=0.05,
        output_dir=DATASET_DIR,
        seed=42,
    )

### Quick dataset preview

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sample = np.load(f'{DATASET_DIR}/traj_0000.npz')
frames = sample['frames']   # (T, H, W, 3)

fig, axes = plt.subplots(1, 8, figsize=(16, 2))
fig.patch.set_facecolor('#111')
for ax, idx in zip(axes, np.linspace(0, len(frames)-1, 8, dtype=int)):
    ax.imshow(frames[idx])
    ax.axis('off')
    ax.set_title(f't={idx}', color='white', fontsize=8)
plt.suptitle('Trajectory 0 — 8 frames', color='white')
plt.tight_layout()
plt.show()

## 5. Training

| Hyperparameter | Value | Note |
|---|---|---|
| `embed_dim` | 128 | latent dimension |
| `hidden_dim` | 512 | MLP transition FFN |
| `lam` | 1.0 | SIGReg weight — renforcé pour tenir face au rollout long |
| `ema_momentum` | 0.996 | target encoder EMA (τ) |
| `rollout_k` | 20 | horizon de prédiction (1 s ≈ demi-période du pendule) |
| `mse_coef` | 0.1 | contrainte de magnitude dans la pred loss |
| `norm_coef` | 0.0 | désactivé — cause collapse avec rollout_k élevé |
| `seq_len` | 100 | window tirée aléatoirement dans 500 frames |
| `epochs` | 50 | |
| `batch_size` | 32 | fits T4 16 GB |
| `lr` | 1e-4 | AdamW + warmup 5ep + cosine |

**Pourquoi `rollout_k=20` :** avec dt=0.05 s, 20 steps = 1 s ≈ demi-période du pendule.  
Le MLP doit prédire une trajectoire complète, ce qui force ω dans z dès le step 1.

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
EMBED_DIM    = 128
HIDDEN_DIM   = 512
LAM          = 1.0    # SIGReg weight
EMA_MOMENTUM = 0.996  # target encoder EMA momentum (τ)
ROLLOUT_K    = 20     # horizon de prédiction
MSE_COEF     = 0.1    # poids du terme MSE dans la pred loss
NORM_COEF    = 0.0    # désactivé
SEQ_LEN      = 100    # fenêtre tirée aléatoirement dans chaque trajectoire
N_PROJ       = 512
EPOCHS       = 50
BATCH_SIZE   = 32
LR           = 1e-4
WEIGHT_DECAY = 0.05
NUM_WORKERS  = 2      # Colab has 2 CPU cores
CKPT_DIR     = 'checkpoints'
VIS_DIR      = 'visuals'
RESUME       = None   # set to 'checkpoints/jepa/lewm_best.pt' to resume

In [ ]:
import time, math
from pathlib import Path

import torch
import torch.optim as optim
import matplotlib.pyplot as plt

from models.jepa.model import LeWorldModel
from data.dataset import make_seq_dataloaders

device = torch.device('cuda')
print(f'Training on: {torch.cuda.get_device_name(0)}')

# ── Dataloaders ────────────────────────────────────────────────────────────────
train_loader, val_loader = make_seq_dataloaders(
    dataset_dir=DATASET_DIR,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    seq_len=SEQ_LEN,
)
print(f'Train: {len(train_loader)} batches  |  Val: {len(val_loader)} batches')

# ── Model ──────────────────────────────────────────────────────────────────────
model = LeWorldModel(
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    lam=LAM,
    mse_coef=MSE_COEF,
    norm_coef=NORM_COEF,
    n_proj=N_PROJ,
    ema_momentum=EMA_MOMENTUM,
    rollout_k=ROLLOUT_K,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')

# ── Optimizer & scheduler ──────────────────────────────────────────────────────
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def lr_lambda(epoch):
    warmup = 5
    if epoch < warmup:
        return (epoch + 1) / warmup
    t = (epoch - warmup) / max(1, EPOCHS - warmup)
    return 0.5 * (1 + math.cos(math.pi * t))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Evaluate ───────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    sums = {'loss': 0.0, 'pred_loss': 0.0, 'sigreg': 0.0}
    for frames, _ in loader:
        m = model(frames.to(device, non_blocking=True))
        for k in sums:
            sums[k] += m[k].item()
    n = len(loader)
    return {k: v / n for k, v in sums.items()}

# ── Resume ─────────────────────────────────────────────────────────────────────
start_epoch = 1
if RESUME and Path(RESUME).exists():
    ckpt = torch.load(RESUME, map_location=device)
    model.load_state_dict(ckpt['model'], strict=False)
    optimizer.load_state_dict(ckpt['optimizer'])
    start_epoch = ckpt['epoch'] + 1
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda, last_epoch=ckpt['epoch'])
    print(f'Resumed from epoch {ckpt["epoch"]}')

Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)
Path(VIS_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────

history  = {k: [] for k in ['train', 'val', 'pred', 'sigreg']}
best_val = float('inf')

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    t0   = time.time()
    sums = {'loss': 0.0, 'pred_loss': 0.0, 'sigreg': 0.0}

    for frames, _ in train_loader:
        frames = frames.to(device, non_blocking=True)
        optimizer.zero_grad()
        m = model(frames)
        m['loss'].backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        model.update_target()
        for k in sums:
            sums[k] += m[k].item()

    scheduler.step()
    n          = len(train_loader)
    train_loss = sums['loss'] / n
    val_m      = evaluate(model, val_loader)

    history['train'].append(train_loss)
    history['val'].append(val_m['loss'])
    history['pred'].append(sums['pred_loss'] / n)
    history['sigreg'].append(sums['sigreg'] / n)

    improved = val_m['loss'] < best_val
    if improved:
        best_val = val_m['loss']
        torch.save({
            'epoch':     epoch,
            'model':     model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'val_loss':  best_val,
            'args': {
                'embed_dim':    EMBED_DIM,
                'hidden_dim':   HIDDEN_DIM,
                'lam':          LAM,
                'mse_coef':     MSE_COEF,
                'norm_coef':    NORM_COEF,
                'n_proj':       N_PROJ,
                'ema_momentum': EMA_MOMENTUM,
                'rollout_k':    ROLLOUT_K,
            },
        }, f'{CKPT_DIR}/lewm_best.pt')

    elapsed = time.time() - t0
    lr_now  = optimizer.param_groups[0]['lr']
    print(
        f'Epoch {epoch:3d}/{EPOCHS}'
        f'  loss={train_loss:.4f}'
        f'  pred={sums["pred_loss"]/n:.4f}'
        f'  sig={sums["sigreg"]/n:.4f}'
        f'  val={val_m["loss"]:.4f}'
        f'  lr={lr_now:.2e}'
        f'  {elapsed:.1f}s'
        + ('  <-- best' if improved else '')
    )

print(f'\nBest val loss: {best_val:.4f}  ->  {CKPT_DIR}/lewm_best.pt')

## 6. Loss curves

In [ ]:
epochs_x = range(1, len(history['train']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#111')

# ── Total loss ─────────────────────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['train'], color='#4fc3f7', label='train')
ax.plot(epochs_x, history['val'],   color='#ff8a65', label='val')
ax.set_title('Total loss', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

# ── Pred + SIGReg ──────────────────────────────────────────────────────────────
ax = axes[1]
ax.set_facecolor('#111')
ax.plot(epochs_x, history['pred'],   color='#a5d6a7', label='pred loss (cosine+MSE)')
ax.plot(epochs_x, history['sigreg'], color='#ce93d8', label='SIGReg')
ax.set_title('Pred + SIGReg', color='white')
ax.set_xlabel('epoch', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig(f'{VIS_DIR}/lewm_loss_colab.png', dpi=100, bbox_inches='tight', facecolor='#111')
plt.show()

## 7. Latent space visualisation (PCA)

JEPA ne peut pas décoder en pixels — on visualise à la place la **structure de l'espace latent**.

- Chaque point = embedding z d'une frame
- Couleur = progression temporelle dans la trajectoire (bleu → rouge)
- **Si le modèle a bien appris :** les trajectoires forment des courbes lisses et séparées dans l'espace PCA — signature que z encode θ ET ω
- **Si effondrement :** tous les points sont concentrés, les trajectoires se superposent

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# ── Load best checkpoint ───────────────────────────────────────────────────────
ckpt = torch.load(f'{CKPT_DIR}/lewm_best.pt', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded checkpoint from epoch {ckpt["epoch"]}  (val_loss={ckpt["val_loss"]:.4f})')

# ── Encode a few val trajectories ─────────────────────────────────────────────
N_TRAJS  = 8   # nombre de trajectoires à visualiser
all_z    = []
all_time = []
all_traj = []

with torch.no_grad():
    for i, (frames, _) in enumerate(val_loader):
        if i >= N_TRAJS:
            break
        z = model.encode(frames[:1].to(device))   # (1, T, D)
        T = z.shape[1]
        all_z.append(z[0].cpu().numpy())           # (T, D)
        all_time.append(np.linspace(0, 1, T))
        all_traj.append(np.full(T, i))

Z    = np.concatenate(all_z,    axis=0)   # (N_TRAJS*T, D)
time = np.concatenate(all_time, axis=0)
traj = np.concatenate(all_traj, axis=0)

# ── PCA (2D) ───────────────────────────────────────────────────────────────────
Z_centered = Z - Z.mean(0)
_, _, Vt   = np.linalg.svd(Z_centered, full_matrices=False)
PC         = Z_centered @ Vt[:2].T    # (N, 2)

var_explained = np.var(PC, axis=0) / np.var(Z_centered, axis=None)
print(f'PC1 var explained: {var_explained[0]:.1%}  |  PC2: {var_explained[1]:.1%}')

# ── Plot ───────────────────────────────────────────────────────────────────────
cmap   = get_cmap('plasma')
colors = ['#4fc3f7', '#ff8a65', '#a5d6a7', '#ce93d8',
          '#ffcc80', '#ef9a9a', '#80cbc4', '#f48fb1']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#111')

# Panel 1 : coloré par trajectoire
ax = axes[0]
ax.set_facecolor('#111')
for i in range(N_TRAJS):
    mask = traj == i
    ax.plot(PC[mask, 0], PC[mask, 1], color=colors[i], alpha=0.7, lw=1.2, label=f'traj {i}')
    ax.scatter(PC[mask, 0][0],  PC[mask, 1][0],  color=colors[i], s=60, marker='o', zorder=5)  # start
    ax.scatter(PC[mask, 0][-1], PC[mask, 1][-1], color=colors[i], s=60, marker='x', zorder=5)  # end
ax.set_title('PCA — trajectoires (o=start, x=end)', color='white')
ax.set_xlabel('PC1', color='white')
ax.set_ylabel('PC2', color='white')
ax.legend(facecolor='#222', labelcolor='white', edgecolor='#444', fontsize=7)
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

# Panel 2 : coloré par progression temporelle (toutes traj confondues)
ax = axes[1]
ax.set_facecolor('#111')
sc = ax.scatter(PC[:, 0], PC[:, 1], c=time, cmap='plasma', s=4, alpha=0.6)
cbar = plt.colorbar(sc, ax=ax)
cbar.ax.yaxis.set_tick_params(color='white')
cbar.set_label('t/T', color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')
ax.set_title('PCA — progression temporelle', color='white')
ax.set_xlabel('PC1', color='white')
ax.set_ylabel('PC2', color='white')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')

plt.tight_layout()
plt.savefig(f'{VIS_DIR}/lewm_latent_pca.png', dpi=120, bbox_inches='tight', facecolor='#111')
plt.show()

## 8. Download checkpoint

In [ ]:
from google.colab import files
files.download(f'{CKPT_DIR}/lewm_best.pt')